In [2]:
import re
from pathlib import Path

archivo_entrada = Path("transport.m3d")
archivo_salida = Path("transport_dos_especies.txt")
ncol = 31  # número de celdas (columnas)


In [3]:
def extraer_concentraciones_desde_m3d(archivo_entrada, archivo_salida, ncol):
    with archivo_entrada.open("r") as f:
        lineas = f.readlines()

    bloques = {}
    componente_actual = None
    tiempo_actual = None
    leyendo_datos = False
    datos_temporales = []

    for i, linea in enumerate(lineas):
        if "FOR COMPONENT NO." in linea:
            componente_actual = int(re.search(r"COMPONENT NO\.\s*(\d+)", linea).group(1))
            bloques[componente_actual] = {}

        elif "TOTAL ELAPSED TIME SINCE BEGINNING OF SIMULATION" in linea:
            match = re.search(r"=\s*([0-9.Ee+-]+)", linea)
            if match:
                tiempo_actual = float(match.group(1))
            else:
                tiempo_actual = None

        elif "CONCENTRATIONS" in linea:
            leyendo_datos = True
            datos_temporales = []

        elif leyendo_datos:
            if not linea.strip() or "." not in linea:
                # fin del bloque
                leyendo_datos = False
                if tiempo_actual is not None and componente_actual is not None:
                    flat = re.findall(r"[0-9.Ee+-]+", " ".join(datos_temporales))
                    if len(flat) > 1:
                        concentraciones = [float(x) for x in flat[1:ncol+1]]
                        bloques[componente_actual][tiempo_actual] = concentraciones
            else:
                datos_temporales.append(linea.strip())

    # Escribir al archivo de salida
    with archivo_salida.open("w") as f:
        for comp in sorted(bloques.keys()):
            f.write(f"# Componente {comp}\n")
            for tiempo in sorted(bloques[comp].keys()):
                f.write(f"> Tiempo: {tiempo:.6f} días\n")
                concs = bloques[comp][tiempo]
                for i, val in enumerate(concs, 1):
                    f.write(f"{val:.8e} ")
                    if i % 10 == 0 or i == len(concs):
                        f.write("\n")
            f.write("\n")

    print(f"✅ Archivo generado: {archivo_salida}")


In [4]:
extraer_concentraciones_desde_m3d(archivo_entrada, archivo_salida, ncol)


✅ Archivo generado: transport_dos_especies.txt
